# 08 · PDL — when the profiler proves you wrong

This is the most useful notebook in the series, because the hypothesis was wrong.

Programmatic Dependent Launch (PDL) lets a kernel signal that its dependents may start before it has finished. `fp16_gemm_6` fuses a dequantisation kernel with a GEMM: dequant loads B from DRAM, dequantises, and calls `pdl_launch_dependents`; the GEMM launches concurrently and reaches `pdl_wait` before consuming B.

**Expected:** meaningful overlap between dequant execution and GEMM prologue. **Measured:** almost none — and the reason is more instructive than the win would have been.

## 1. The question

Two kernels with a real data dependency:

1. **dequant** — loads B, dequantises, writes it, fires `pdl_launch_dependents`
2. **GEMM** — launches early, runs its preamble, blocks at `pdl_wait` until B is ready

**The question:** does PDL actually produce overlap, and what bounds it?

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        610.57.04
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

Both kernels are annotated, because the interesting quantity is a *relationship between* them — when one signals versus when the other starts:

```python
# dequant kernel
iket.range_push("dequant_kernel")
iket.range_push("dequant_load");  ...; iket.range_pop()
iket.range_push("pdl_launch_dependents"); iket.range_pop()
iket.range_push("dequant_store"); ...; iket.range_pop()
iket.range_pop()

# GEMM kernel — TMA warp
iket.range_push("tma_block")
iket.range_push("pdl_wait");  ...; iket.range_pop()
iket.range_pop()
```

### A wrinkle worth knowing

`fp16_gemm_6` is **CUDA-graph launched**. Graph-launched kernels do not appear under `launches` in the JSON trace — their events live under `graphLaunches`, keyed by graph exec.

Read only `launches` and this trace looks *empty*, which is easy to misdiagnose as trace-buffer overflow. `trace_summary.all_launches()` merges both.

We check that explicitly once the trace exists, a few cells below.

## 3. Run it

Two shapes, chosen to differ in one property: how many SMs the dequant kernel leaves free.

In [2]:
for tag, mnk in [("8192", "256,8192,128"), ("18944", "256,18944,128")]:
    cmd = (
        f"run-iket --output-dir out/nb08_gemm6_{tag} --clobber profile --postprocess all -- "
        f"python {nbhelp.kernel('instrumented', 'fp16_gemm_6_iket_detailed.py')} --mnk {mnk}"
    )
    print(f"===== {tag} =====")
    check_env.run_or_show(cmd, env, est="about 1 minute each")

===== 8192 =====
$ run-iket --output-dir out/nb08_gemm6_8192 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_6_iket_detailed.py --mnk 256,8192,128



[run-iket] Removing existing output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb08_gemm6_8192
[run-iket] ============= Dry run application first to collect some info... =============
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_6_iket_detailed.py:816: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
Running Blackwell fp16 GEMM example 6 with:
  mnk:       [256, 8192, 128]
  tolerance: 0.1

PASS
[run-iket] Auto-computed context buffer size: 0.00 GB (1,308,313 bytes)
[run-iket] ============= Run application for real profiling... =============
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_6_iket_detailed.py:816: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be remov

[run-iket] Removing existing output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb08_gemm6_18944
[run-iket] ============= Dry run application first to collect some info... =============
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_6_iket_detailed.py:816: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
Running Blackwell fp16 GEMM example 6 with:
  mnk:       [256, 18944, 128]
  tolerance: 0.1

PASS
[run-iket] Auto-computed context buffer size: 0.00 GB (3,025,475 bytes)
[run-iket] ============= Run application for real profiling... =============
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_6_iket_detailed.py:816: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be rem

## 4. The trace

First, the `graphLaunches` point from above — this is what an empty-looking trace that is actually fine looks like:

In [3]:
try:
    trace = ts.load(ts.find_trace("out/nb08_gemm6_8192"))
    print("launches      :", len(trace.get("launches", [])))
    print("graphLaunches :", sum(len(v) for v in trace.get("graphLaunches", {}).values()))
    print("merged        :", len(ts.all_launches(trace)))
    print()
    print("Reading only 'launches' would report an empty trace here.")
except FileNotFoundError:
    print("No local trace — reference: launches=0, graphLaunches=2, merged=2.")

launches      : 0
graphLaunches : 2
merged        : 2

Reading only 'launches' would report an empty trace here.


In [4]:
nbhelp.figure("nb08/gemm6_pdl_overlap_8192.png", "256x8192x128: dequant launches 128 CTAs, leaving 20 of 148 SMs idle — the GEMM starts there at once.")

In [5]:
nbhelp.figure("nb08/gemm6_pdl_overlap_18944.png", "256x18944x128: 296 CTAs cover every SM, so the GEMM packs in tightly behind dequant.")

In [6]:
nbhelp.figure("nb08/gemm6_pdl_full_occupancy_timeline.png", "At full occupancy: launch_dependents fires at 4.1 us, but the GEMM cannot start until 11.4 us.")

In [7]:
try:
    trace = ts.load(ts.find_trace("out/nb08_gemm6_8192"))
    ts.print_summary(ts.summarize(trace), top=8)
except FileNotFoundError:
    print("No local trace here — the reference numbers below were measured on a NVIDIA B300.")

kernel_cutlass_dequantize_tensorptri8gmemalign32odiv128i64div1281__tensorptrf16gmemalign32odiv128i64div1281_0   grid 128x1x1  block 64x1x1  warps traced 256
warp-lifetime span: 2.2 us

range                      count      median         p90       total
--------------------------------------------------------------------
dequant_kernel               256      1.664u      1.920u     423.58u
dequant_store                256      1.248u      1.536u     322.34u
dequant_load                 256      0.160u      0.192u      44.83u
dequant_setup                256      0.032u      0.064u      10.56u
pdl_launch_dependents        256      0.032u      0.032u       6.91u


## 5. The finding

Measured on a NVIDIA B300, dequant kernel:

| Range | 256×8192×128 | 256×18944×128 |
|---|---|---|
| `dequant_kernel` | 1.664 µs | 1.664 µs |
| `dequant_store` | 1.280 µs | 1.248 µs |
| `dequant_load` | 0.160 µs | 0.160 µs |
| Warps traced | 256 | 592 |


At 4096×4096×4096, where the effect was studied in most detail:

- `launch_dependents` fires at **4.1 µs**, but the dequant kernel runs on to 12.6 µs
- PDL makes the GEMM *eligible* early — yet it cannot start until **11.4 µs**, because every SM is still busy with dequant
- Realised saving: **1.25 µs (1.4%)** — 89.3 µs with PDL versus 90.6 µs serial


### Why it failed, and why that is the useful part

PDL does not create capacity. It removes a *scheduling* barrier, letting dependent CTAs launch and run their preamble early. But a CTA still needs an SM to run on.

- At 256×8192×128, dequant uses 128 CTAs and leaves 20 SMs idle. The GEMM starts there immediately — genuine overlap, bounded by those 20 SMs.
- At 256×18944×128, 296 CTAs cover every SM. There is nowhere for the GEMM to go, so PDL buys essentially nothing.


**The overlap window is set by idle-SM capacity, and its ceiling by preamble length.** Work that consumes the producer's output always waits for `pdl_wait` regardless.

Two shapes, opposite causes, same lesson: the mechanism worked exactly as documented, and the expected benefit still did not materialise, because the binding constraint was somewhere else entirely. A wall-clock A/B would have shown 1.4% and left you guessing. The trace showed *which microsecond* the GEMM was waiting for and *what* it was waiting on.

**This is what a profiler is for.** Confirming a model is pleasant; breaking one is where the value is.

**Next:** [09 · The optimization ladder](09_optimization_ladder.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*